# 05 - Security perspective

What the binary models would mean for a SOC: which normal flows raise false alarms, which attacks get through, and how many alerts a day that would be. Nothing here changes a model or a threshold; the random forest is refit with the hyperparameters and threshold saved by `02_supervised`, and the other numbers come from `results.json`. Everything is read on test, so it is diagnostic.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline

from netad.data import load_unsw_frame, train_val_split, distinct_rows, FEATURES
from netad.preprocessing import build_preprocessor
from netad.evaluation import RESULTS_PATH, results_table

In [3]:
train_df = load_unsw_frame("train")
test_df = load_unsw_frame("test")
X_tr, X_val, y_tr, y_val = train_val_split(train_df, "label")

saved = {(r["model"], r["task"]): r for r in json.loads(RESULTS_PATH.read_text(encoding="utf-8"))}
entry = saved[("RandomForest", "binary")]
rf_params = {k: v for k, v in entry["params"].items() if k in RandomForestClassifier().get_params()}
rf = make_pipeline(build_preprocessor(), RandomForestClassifier(random_state=42, n_jobs=-1, **rf_params)).fit(X_tr, y_tr)
threshold = entry["test"]["threshold"]

test = test_df.assign(score=rf.predict_proba(test_df[FEATURES])[:, 1])
test["flagged"] = test["score"] >= threshold
test["distinct"] = distinct_rows(test_df)
normal, attacks = test[test["label"] == 0], test[test["label"] == 1]
fp, fn = normal[normal["flagged"]], attacks[~attacks["flagged"]]

# same numbers as 02_supervised
pd.Series({"recall": attacks["flagged"].mean(), "FPR": normal["flagged"].mean(),
           "false positives": len(fp), "false negatives": len(fn)}).round(4)

recall                0.8905
FPR                   0.0397
false positives    1470.0000
false negatives    4963.0000
dtype: float64

## False positives

Normal test flows flagged as attack. For each value: its share of the false positives, its share of all normal traffic, and the FPR within it.

In [4]:
def breakdown(flagged_rows, all_rows, col, top=6):
    return pd.DataFrame({
        "share of errors": flagged_rows[col].value_counts(normalize=True),
        "share of class": all_rows[col].value_counts(normalize=True),
        "rate": all_rows.groupby(col)["flagged"].mean(),
    }).sort_values("share of errors", ascending=False).head(top)


pd.concat({col: breakdown(fp, normal, col).round(3) for col in ["proto", "service", "state", "sttl"]})

share of errors  share of class   rate
proto   tcp                 0.598           0.753  0.032
        udp                 0.400           0.219  0.073
        igmp                0.002           0.001  0.100
        arp                   NaN           0.027  0.000
        ospf                  NaN           0.001  0.000
service -                   0.844           0.740  0.045
        http                0.154           0.108  0.056
        radius              0.001           0.000  1.000
        ftp                 0.001           0.020  0.001
        dns                 0.001           0.083  0.000
        ftp-data              NaN           0.026  0.000
state   FIN                 0.598           0.653  0.036
        INT                 0.396           0.121  0.130
        REQ                 0.006           0.046  0.005
        ACC                   NaN           0.000  0.000
        CON                   NaN           0.179  0.000
        RST                   NaN           0.000  0.000
sttl    254                 0.870           0.407  0.085
        62                  0.130           0.112  0.046
        0                     NaN           0.027  0.000
        1                     NaN           0.001  0.000
        29                    NaN           0.001  0.000
        30                    NaN           0.000  0.000

In [5]:
fp.groupby(["proto", "service", "state", "sttl"]).size().sort_values(ascending=False).head(8).rename("false positives")

proto  service  state  sttl
tcp    -        FIN    254     647
udp    -        INT    254     576
tcp    http     FIN    62      186
                       254      40
udp    -        REQ    254       9
tcp    -        FIN    62        5
igmp   -        INT    254       3
udp    radius   INT    254       2
Name: false positives, dtype: int64

A sample of 15 distinct false positives, and the median of a few columns for false positives against correctly passed normal flows:

In [6]:
COLS = ["proto", "service", "state", "sttl", "dttl", "dur", "spkts", "dpkts", "sbytes", "dbytes", "ct_srv_src", "score"]
fp[fp["distinct"]].sample(15, random_state=42)[COLS].round(3)

,proto,service,state,sttl,dttl,dur,spkts,dpkts,sbytes,dbytes,ct_srv_src,score
66961,udp,-,INT,254,0,0.000,2,0,104,0,2,0.952
75717,tcp,-,FIN,254,252,1.327,12,10,984,816,1,0.911
69623,tcp,-,FIN,254,252,1.972,14,12,774,642,1,0.898
67558,tcp,-,FIN,254,252,1.201,12,8,1106,712,5,0.907
76909,tcp,-,FIN,254,252,0.544,10,6,588,268,2,0.885
73579,tcp,-,FIN,254,252,2.161,12,10,994,804,2,0.927
67076,tcp,-,FIN,254,252,1.200,10,6,926,268,3,0.925
73399,tcp,-,FIN,254,252,2.617,14,16,1284,1396,2,0.873
69630,tcp,-,FIN,254,252,0.589,12,8,1162,726,3,0.963
40988,udp,-,INT,254,0,0.000,2,0,212,0,4,0.871


In [7]:
pd.DataFrame({
    "false positives": fp[COLS].median(numeric_only=True),
    "normal, not flagged": normal[~normal["flagged"]][COLS].median(numeric_only=True),
}).round(3)

,false positives,"normal, not flagged"
sttl,254.000,31.000
dttl,252.000,29.000
dur,0.646,0.210
spkts,10.000,10.000
dpkts,8.000,8.000
sbytes,794.000,978.000
dbytes,354.000,354.000
ct_srv_src,2.000,4.000
score,0.909,0.001


## False negatives

Attacks that pass. Share of the missed attacks per category and the miss rate within each category:

In [8]:
pd.DataFrame({
    "share of misses": fn["attack_cat"].value_counts(normalize=True),
    "missed": 1 - attacks.groupby("attack_cat")["flagged"].mean(),
    "missed, distinct rows": 1 - attacks[attacks["distinct"]].groupby("attack_cat")["flagged"].mean(),
}).sort_values("share of misses", ascending=False).round(3)

,share of misses,missed,"missed, distinct rows"
attack_cat,,,
Fuzzers,0.755,0.618,0.697
Exploits,0.130,0.058,0.072
DoS,0.036,0.043,0.081
Generic,0.029,0.008,0.024
Reconnaissance,0.027,0.038,0.046
Analysis,0.012,0.086,0.130
Shellcode,0.010,0.130,0.130
Backdoor,0.001,0.012,0.020
Worms,0.001,0.068,0.068


In [9]:
fn.groupby(["attack_cat", "proto", "service", "state"]).size().sort_values(ascending=False).head(8).rename("missed")

attack_cat      proto  service  state
Fuzzers         tcp    -        FIN      2361
                udp    -        INT       945
Exploits        tcp    -        FIN       426
Fuzzers         tcp    ftp      FIN       264
                       http     FIN       161
Exploits        tcp    http     FIN       144
Reconnaissance  tcp    -        FIN        79
DoS             tcp    -        FIN        78
Name: missed, dtype: int64

In [10]:
fn[fn["distinct"]].sample(15, random_state=42)[["attack_cat"] + COLS].round(3)

,attack_cat,proto,service,state,sttl,dttl,dur,spkts,dpkts,sbytes,dbytes,ct_srv_src,score
9076,Exploits,tcp,-,FIN,254,252,0.771,10,10,608,646,3,0.554
50456,Fuzzers,tcp,ftp,FIN,254,252,1.029,12,12,826,682,1,0.829
8122,Fuzzers,tcp,ftp,FIN,254,252,0.574,12,12,664,682,3,0.375
51839,Fuzzers,tcp,-,FIN,254,252,0.521,10,8,534,354,3,0.733
45425,Fuzzers,tcp,-,FIN,254,252,0.451,10,6,2062,268,5,0.831
1230,Fuzzers,tcp,-,FIN,254,252,1.406,12,10,1494,848,3,0.754
44162,Exploits,tcp,http,FIN,62,252,1.959,10,10,844,1310,1,0.420
43887,Fuzzers,tcp,-,FIN,254,252,0.324,10,6,486,268,12,0.368
52415,Fuzzers,udp,-,INT,254,0,0.000,2,0,104,0,12,0.793
48431,Fuzzers,tcp,-,FIN,254,252,0.451,16,10,900,556,7,0.278


Fuzzers is three quarters of the misses. Missed and caught Fuzzers flows, next to the false positives:

In [11]:
fuzzers = attacks[attacks["attack_cat"] == "Fuzzers"]
pd.DataFrame({
    "Fuzzers missed": fuzzers[~fuzzers["flagged"]][COLS].median(numeric_only=True),
    "Fuzzers caught": fuzzers[fuzzers["flagged"]][COLS].median(numeric_only=True),
    "false positives": fp[COLS].median(numeric_only=True),
}).round(3)

,Fuzzers missed,Fuzzers caught,false positives
sttl,254.000,254.000,254.000
dttl,252.000,0.000,252.000
dur,0.566,0.000,0.646
spkts,10.000,2.000,10.000
dpkts,6.000,0.000,8.000
sbytes,764.000,200.000,794.000
dbytes,268.000,0.000,354.000
ct_srv_src,6.000,6.000,2.000
score,0.684,0.957,0.909


In [12]:
pd.concat({
    "Fuzzers missed": fuzzers[~fuzzers["flagged"]][["proto", "state"]].value_counts(normalize=True).head(3),
    "Fuzzers caught": fuzzers[fuzzers["flagged"]][["proto", "state"]].value_counts(normalize=True).head(3),
    "false positives": fp[["proto", "state"]].value_counts(normalize=True).head(3),
}).round(3)

                 proto  state
Fuzzers missed   tcp    FIN      0.743
                 udp    INT      0.252
                 tcp    CON      0.002
Fuzzers caught   tcp    FIN      0.396
                 unas   INT      0.230
                 udp    INT      0.057
false positives  tcp    FIN      0.598
                 udp    INT      0.394
                        REQ      0.006
Name: proportion, dtype: float64

## Alerts per day

A hypothetical sensor seeing 1M flows a day. Real networks have far fewer attacks than the test file (55% of its rows), so two base rates are used: 1% and 0.1% of flows are attacks. FPR is read on the full test set (each flagged normal flow is an alert), recall on the distinct test rows so that repeated easy flows don't inflate it. The thresholds are the ones picked on validation, as they would be deployed.

In [13]:
FLOWS_PER_DAY = 1_000_000
models = ["LogisticRegression", "RandomForest", "HistGradientBoosting", "IsolationForest", "Autoencoder"]
fpr = results_table("binary", "test", ["fpr"]).loc[models, "fpr"]
recall = results_table("binary", "test_distinct", ["recall"]).loc[models, "recall"]

rows = {}
for base_rate in [0.01, 0.001]:
    n_attack = FLOWS_PER_DAY * base_rate
    n_normal = FLOWS_PER_DAY - n_attack
    for name in models:
        false_alerts, true_alerts = n_normal * fpr[name], n_attack * recall[name]
        rows[(f"{base_rate:.1%} attacks", name)] = {
            "test FPR": fpr[name], "recall (distinct)": recall[name],
            "alerts": false_alerts + true_alerts, "false alerts": false_alerts,
            "attacks caught": true_alerts, "attacks missed": n_attack - true_alerts,
            "precision": true_alerts / (true_alerts + false_alerts),
        }
alerts = pd.DataFrame(rows).T
alerts.round({"test FPR": 4, "recall (distinct)": 3, "precision": 3, "alerts": 0, "false alerts": 0,
              "attacks caught": 0, "attacks missed": 0})

test FPR  recall (distinct)   alerts  \
1.0% attacks LogisticRegression      0.0252              0.631  31225.0   
             RandomForest            0.0397              0.798  47317.0   
             HistGradientBoosting    0.0481              0.855  56148.0   
             IsolationForest         0.0264              0.295  29060.0   
             Autoencoder             0.0312              0.304  33943.0   
0.1% attacks LogisticRegression      0.0252              0.631  25768.0   
             RandomForest            0.0397              0.798  40488.0   
             HistGradientBoosting    0.0481              0.855  48888.0   
             IsolationForest         0.0264              0.295  26647.0   
             Autoencoder             0.0312              0.304  31489.0   

                                   false alerts  attacks caught  \
1.0% attacks LogisticRegression         24911.0          6314.0   
             RandomForest               39332.0          7984.0   
             HistGradientBoosting       47600.0          8547.0   
             IsolationForest            26115.0          2945.0   
             Autoencoder                30904.0          3039.0   
0.1% attacks LogisticRegression         25137.0           631.0   
             RandomForest               39690.0           798.0   
             HistGradientBoosting       48033.0           855.0   
             IsolationForest            26352.0           295.0   
             Autoencoder                31185.0           304.0   

                                   attacks missed  precision  
1.0% attacks LogisticRegression            3686.0      0.202  
             RandomForest                  2016.0      0.169  
             HistGradientBoosting          1453.0      0.152  
             IsolationForest               7055.0      0.101  
             Autoencoder                   6961.0      0.090  
0.1% attacks LogisticRegression             369.0      0.025  
             RandomForest                   202.0      0.020  
             HistGradientBoosting           145.0      0.017  
             IsolationForest                705.0      0.011  
             Autoencoder                    696.0      0.010

In [14]:
# what was planned (1% FPR, as calibrated on validation) against what the test FPR gives
pd.Series({
    "false alerts planned (1% FPR)": FLOWS_PER_DAY * 0.99 * 0.01,
    "false alerts at test FPR (random forest)": alerts.loc[("1.0% attacks", "RandomForest"), "false alerts"],
}).round(0)

false alerts planned (1% FPR)                9900.0
false alerts at test FPR (random forest)    39332.0
dtype: float64

## Conclusions

1. **The false alarms are concentrated in normal flows with sttl 254.** They are 87% of the false positives and 41% of normal test traffic (19% on validation). By shape: TCP connections with no recognised service closed with FIN (60% of false positives), and one-way UDP flows in state INT (40%, FPR 13% within that state). As in `02`, this is an association: the model without TTL splits normal traffic the same way.
2. **Fuzzers is three quarters of what gets through.** 70% of distinct Fuzzers flows are missed. The missed ones have the same profile as the false positives: TCP, no service, FIN, sttl 254, ~10 packets in ~0.6 s, a few hundred bytes each way. The caught ones are 2-packet flows with no answer. At flow level those Fuzzers and that part of the normal traffic overlap; telling them apart would need payload or the sequence of a host's flows.
3. **Alert volume makes a per-flow classifier unusable on its own.** At 1M flows a day the random forest's 4.0% test FPR gives ~39,000 false alerts instead of the ~9,900 planned at 1%. With 1% of flows being attacks, 17% of its alerts are real; at 0.1%, closer to a real network, 2%, about 50 false alerts per true one. The other models move along the same trade-off (1% attacks): logistic regression has ~25,000 false alerts, catches 63% and is right 20% of the time; gradient boosting ~48,000, 86% and 15%; the anomaly detectors ~26,000-31,000 false alerts for 30% caught, 9-10% precision. Flows would have to be grouped by host and time window and ranked before an analyst sees them.